In [5]:
%%capture
!pip install -q git+https://github.com/huggingface/transformers.git accelerate pdf2image pillow requests datasets
!apt-get install -q poppler-utils

In [6]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

CUDA available: True
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [7]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from PIL import Image
import requests
import torch

model_id = "Qwen/Qwen3.5-9B"
print("Loading Qwen3.5-9B...")

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16
)

# Load and resize the image to prevent OOM
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
image = Image.open(requests.get(url, stream=True).raw)
image.thumbnail((1024, 1024))

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading Qwen3.5-9B...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


The user wants to identify the animal depicted on the candies in the image.

1.  **Analyze the image:** I see a hand holding five small, round candies. They look like jelly


In [8]:
import os
print("Contents of /kaggle/input:")
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        print(f"ROOT: {root}")
        for d in dirs:
            print(f"  DIR: {d}")
        for f in files:
            print(f"  FILE: {f}")
else:
    print("/kaggle/input does not exist")

Contents of /kaggle/input:
ROOT: /kaggle/input
  DIR: datasets
ROOT: /kaggle/input/datasets
  DIR: rudhrakoul
ROOT: /kaggle/input/datasets/rudhrakoul
  DIR: invoices
ROOT: /kaggle/input/datasets/rudhrakoul/invoices
  DIR: invoice_data
ROOT: /kaggle/input/datasets/rudhrakoul/invoices/invoice_data
  FILE: 4519029070_202223MLKA0281_jpg.rf.nE3rzyZ35ZdFQEpLerFV.jpg
  FILE: GW01210600825-pdf_page_1_png_jpg.rf.fH11JuOMER2lDYZLnKyC.jpg
  FILE: DSINV2023241716-pdf_page_1_png_jpg.rf.wAzHl04I4fAOdAR2MMh0.jpg
  FILE: 5200018764-pdf_page_1_png_jpg.rf.RIZRc3cF6y79nlrmUNH3.jpg
  FILE: CMA-CGM-pdf_page_1_png_jpg.rf.It1DfmILEKK8Lmi3uVaN.jpg
  FILE: Associated-Services-pdf_page_1_png_jpg.rf.lb1HNl0cLIaaUXVnPWfL.jpg
  FILE: Class-III-3-pdf_page_7_png_jpg.rf.tlzcwEgvifTeOZXnB5Rs.jpg
  FILE: 5788040277-pdf_page_1_png_jpg.rf.h3QSK19xzM1E7tutNb1d.jpg
  FILE: DSINV2023241702-pdf_page_2_png_jpg.rf.GJObN4UDda4uKtGzPF75.jpg
  FILE: Logistics-1-pdf_page_1_png_jpg.rf.B0TvPp0mNu2clX6amMyJ.jpg
  FILE: DSINV202324174

In [9]:
# Stage 2 — File Ingestion & Pre-processing
import os
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image

# We search the entire /kaggle/input directory to avoid missing files due to unknown mount paths
INPUT_DIR = "/kaggle/input"

all_files = []
for root, _, files in os.walk(INPUT_DIR):
    for f in files:
        ext = f.lower()
        if ext.endswith(('.pdf', '.png', '.jpg', '.jpeg')):
            all_files.append(os.path.join(root, f))

print(f"Found {len(all_files)} invoice files")

# Quick breakdown by type
from collections import Counter
type_counts = Counter(Path(f).suffix.lower() for f in all_files)
for ext, count in type_counts.items():
    print(f"  {ext}: {count} files")

def pdf_to_images(pdf_path: str, dpi: int = 200) -> list:
    return convert_from_path(pdf_path, dpi=dpi)

def load_image(image_path: str) -> Image.Image:
    return Image.open(image_path).convert("RGB")


Found 640 invoice files
  .jpg: 636 files
  .jpeg: 4 files


In [10]:
# Stage 3 — Inference Logic
SYSTEM_PROMPT = "You are an automated data extraction API. You do not speak. You do not converse. You strictly output raw JSON only."

VENDOR_PROMPT = """You are an invoice data extraction specialist. Your job is to extract the vendor's GSTIN, company name, and full address from this invoice.

Guidelines for GSTIN:
- A GSTIN is exactly 15 characters (e.g. 27AABCT1029L1ZX).
- Search near labels like GSTIN, GST No, Tax ID, etc.
- Extract the vendor/seller's GSTIN, not the buyer's.

Guidelines for Vendor Details:
- vendor_name: The name of the company or person selling the goods/services. This is usually at the very top of the invoice or above the vendor's address.
- vendor_address: The complete physical address of the vendor. Include city, state, and pincode if visible.

Return ONLY this JSON, nothing else:
{
  "gst_number": "the 15 character GSTIN or null if completely absent",
  "gst_label_found": "the exact label text you found near it e.g. GSTIN",
  "gst_location": "where on the invoice you found it e.g. top right header",
  "vendor_name": "exact company name of the vendor",
  "vendor_address": "full address of the vendor"
}"""

def extract_fields(img: Image.Image, source_file: str, page: int) -> tuple:
    img.thumbnail((1024, 1024))
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": VENDOR_PROMPT}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    prefill_text = '{\n  "gst_number":'
    prefill_ids = processor.tokenizer.encode(prefill_text, add_special_tokens=False)
    
    seq_len_before = inputs["input_ids"].shape[1]
    for k, v in inputs.items():
        if isinstance(v, torch.Tensor) and v.ndim == 2 and v.shape[1] == seq_len_before:
            if k == "input_ids":
                pad_tensor = torch.tensor([prefill_ids], dtype=v.dtype, device=model.device)
            elif k == "attention_mask":
                pad_tensor = torch.ones((1, len(prefill_ids)), dtype=v.dtype, device=model.device)
            elif k == "position_ids":
                last_pos = v[0, -1].item()
                pad_tensor = torch.arange(last_pos + 1, last_pos + 1 + len(prefill_ids), dtype=v.dtype, device=model.device).unsqueeze(0)
            else:
                pad_tensor = torch.zeros((1, len(prefill_ids)), dtype=v.dtype, device=model.device)
            inputs[k] = torch.cat([v, pad_tensor], dim=1)

    outputs = model.generate(
        **inputs, 
        max_new_tokens=1024, 
        do_sample=False
    )
    
    raw = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    
    raw = prefill_text + raw
    return raw, source_file, page

In [11]:
# Stage 4 — Post-processing & Output Handlers
import json
import os
import re

# NEW OUTPUT FILE for Full Run
OUTPUT_FILE = "/kaggle/working/gstinfo.jsonl"

def parse_output(raw: str, source_file: str, page: int) -> dict:
    # Remove <think> tags if the model hallucinates them
    if "</think>" in raw:
        raw = raw.split("</think>")[-1]
        
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if match:
        clean = match.group(1)
    else:
        start = raw.find('{')
        end = raw.rfind('}')
        if start != -1 and end != -1 and end > start:
            clean = raw[start:end+1]
        else:
            clean = raw

    try:
        data = json.loads(clean)
    except json.JSONDecodeError:
        data = {
            "raw_output": raw,
            "parse_error": True
        }

    data["_source_file"] = source_file
    data["_page"]        = page
    return data

def load_processed(output_file: str) -> set:
    processed = set()
    if not os.path.exists(output_file):
        return processed
    with open(output_file) as f:
        for line in f:
            try:
                rec = json.loads(line)
                processed.add((rec["_source_file"], rec["_page"]))
            except:
                pass
    return processed

In [12]:
# Stage 5 — Batch Processing Loop
from tqdm.notebook import tqdm

processed = load_processed(OUTPUT_FILE)
print(f"Resuming — {len(processed)} records already done")

# PROCESS ALL FILES
with open(OUTPUT_FILE, "a") as out:
    for filepath in tqdm(all_files):
        ext = Path(filepath).suffix.lower()
        try:
            images = pdf_to_images(filepath) if ext == ".pdf" else [load_image(filepath)]

            for page_num, img in enumerate(images):
                if (filepath, page_num) in processed:
                    continue

                raw, src, pg = extract_fields(img, filepath, page_num)
                record = parse_output(raw, src, pg)

                out.write(json.dumps(record) + "\n")
                out.flush()   # write immediately to save progress

        except Exception as e:
            out.write(json.dumps({
                "_source_file": filepath,
                "_page": -1,
                "error": str(e)
            }) + "\n")
            out.flush()

print(f"Done. Output -> {OUTPUT_FILE}")

Resuming — 0 records already done


  0%|          | 0/640 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.
[transformers] Setting `pad_toke

Done. Output -> /kaggle/working/gstinfo.jsonl


In [14]:
# Stage 6 — Validation & Display
import pandas as pd

records = []
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE) as f:
        for line in f:
            records.append(json.loads(line))

df = pd.DataFrame(records)

if not df.empty:
    print(f"Total records: {len(df)}")
    print(f"Parse errors:  {df.get('parse_error', pd.Series(dtype=bool)).sum()}")
    print(f"Runtime errors:{df['error'].notna().sum() if 'error' in df.columns else 0}")
    print()

    core_fields = ["gst_number", "gst_label_found", "gst_location", "vendor_name", "vendor_address"]
    print("Field coverage:")
    for field in core_fields:
        if field in df.columns:
            filled = df[field].notna() & (df[field] != "null") & (df[field] != "")
            print(f"  {field:20s}: {filled.mean()*100:.1f}%")
    
    print("\n--- EXTRACTED DATA FOR REVIEW ---")
    pd.set_option('display.max_colwidth', None)
    display(df[['_source_file'] + [c for c in core_fields if c in df.columns]].head(15))
else:
    print("No records to validate yet.")

Total records: 640
Parse errors:  54
Runtime errors:0

Field coverage:
  gst_number          : 83.1%
  gst_label_found     : 83.1%
  gst_location        : 83.1%
  vendor_name         : 90.9%
  vendor_address      : 89.5%

--- EXTRACTED DATA FOR REVIEW ---


,_source_file,gst_number,gst_label_found,gst_location,vendor_name,vendor_address
0,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/4519029070_202223MLKA0281_jpg.rf.nE3rzyZ35ZdFQEpLerFV.jpg,27AABCT0897522T,GSTIN,top left header,KORUM INDIA PVT LTD,"Plot No-2/2, 1st Phase, Block-1, Industrial Estate-69/29, Mainly District, Karnataka-571429"
1,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/GW01210600825-pdf_page_1_png_jpg.rf.fH11JuOMER2lDYZLnKyC.jpg,24AABXC0802G1Z0,GSTIN/UIN,top left header,NOBLE SALES CORPORATION,"Dandev Complex, Swaminarayan Marg, RAJULAJ CITY, Mobile No 9824969352 / 9427491252, State Name : Gujarat, Code : 24"
2,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/DSINV2023241716-pdf_page_1_png_jpg.rf.wAzHl04I4fAOdAR2MMh0.jpg,36AABCT3169K122,GSTIN,middle left section,THERMO CABLES LIMITED,"28, NAGARJUNA HILLS, PUNJAGUTTA, HYDERABAD-500082."
3,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/5200018764-pdf_page_1_png_jpg.rf.RIZRc3cF6y79nlrmUNH3.jpg,09AAMPG3197P1Z1,GSTIN/UIN,top left header,Atul Electricals,5 Nirmal Palza Itc Road Saharanpur 24716555
4,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/CMA-CGM-pdf_page_1_png_jpg.rf.It1DfmILEKK8Lmi3uVaN.jpg,27AABCT1029L1ZX,GSTIN,bottom left footer,"CMA CGM SA, C/O. CGM * ONE INTERNATIONAL CENTER","TOWER 3, 8TH FL, SENAPATI BAPAT MARG, ELPHINSTONE WEST MUMBAI 400013 INDIA"
5,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/Associated-Services-pdf_page_1_png_jpg.rf.lb1HNl0cLIaaUXVnPWfL.jpg,None,None,None,ASSOCIATED SERVICES,"House No. B-69/2, 2nd Floor, Durgachak, PO. & PS.: Durgachak, Haldia, Dist: East Midnapur, Pin - 721602"
6,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/Class-III-3-pdf_page_7_png_jpg.rf.tlzcwEgvifTeOZXnB5Rs.jpg,None,None,None,SEEPEX.,IMPERIAL Industrial Logistics GmbH Carl-Bosch-Straße 2-6 45699 Herten Germany
7,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/5788040277-pdf_page_1_png_jpg.rf.h3QSK19xzM1E7tutNb1d.jpg,29AAFCD6369P2ZQ,GSTIN,top header,DOGA INDIA PRIVATE LIMITED,"SANDHYA FARM, RAGHUNATHPURA, DOOSBALLUR - 561 203"
8,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/DSINV2023241702-pdf_page_2_png_jpg.rf.GJObN4UDda4uKtGzPF75.jpg,27AABCT1029L1ZX,GSTIN,bottom right footer,TATA STEEL DOWNSTREAM PRODUCTS LIMITED,"Registered Office Tata Steel Downstream Products Limited, Tata Centre 41, Chowringhee Road, Kolkata 700071"
9,/kaggle/input/datasets/rudhrakoul/invoices/invoice_data/Logistics-1-pdf_page_1_png_jpg.rf.B0TvPp0mNu2clX6amMyJ.jpg,27AAEPB7960N4Z9,GSTIN,top right header,Cement Carriers,"209,Shrimohini Complex 345,Kingsway Nagpur NAGPUR-444001,- Maharashtra"


In [15]:
from IPython.display import FileLink
FileLink(r'gstinfo.jsonl')


/kaggle/working/gstinfo.jsonl